# Импорты

In [1]:
import asyncio
import sys
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI
from transformers import AutoTokenizer

sys.path.append(str(Path.cwd().parent))

from generation import generate_one
from io_utils import load_jsonl
from long_input import prepare_product_input

# Параметры генерации

In [2]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

generation_params = {
    "temperature": 0.5,
    "top_p": 0.9,
    "repetition_penalty": 1.1,
    "max_tokens": 250,
    "seed": 42,
}

# Клиент и данные

In [4]:
client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=120,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

benchmark = load_jsonl("../data/benchmark.jsonl")

len(benchmark)

300

# Обработка длинного входа

In [8]:
started = time.perf_counter()
prepared_benchmark = [
    prepare_product_input(
        product=product,
        tokenizer=tokenizer,
        max_input_tokens=1500,
    )
    for product in benchmark
]
time_s = time.perf_counter() - started

print(f"Общее время, с: {time_s:.4f}")
print(f"Среднее время на карточку, с: {time_s / len(benchmark) :.4f}")

Общее время, с: 18.8192
Среднее время на карточку, с: 0.0627


In [15]:
from long_input import count_input_tokens

num_tokens = [
    count_input_tokens(product, tokenizer)
    for product in prepared_benchmark
]

print(f"Обработано карточек: {len(num_tokens)}")
print(f"Максимальный вход: {max(num_tokens)}")
print(f"Средний вход: {sum(num_tokens) / len(num_tokens) :.2f}")

Обработано карточек: 300
Максимальный вход: 1500
Средний вход: 1188.88


# Генерация одной карточки с замером latency

In [16]:
async def generate_timed(product, client, model_name, generation_params):
    """Генерирует одну карточку и измеряет её latency."""
    started = time.perf_counter()

    result = await generate_one(
        client=client,
        model_name=model_name,
        source_product=product,
        generation_params=generation_params,
        max_attempts=2,
    )

    latency = time.perf_counter() - started

    return result, latency

In [17]:
async def run_concurrent(items, concurrency, generate_one):
    """Отправляет запросы параллельно, не больше concurrency одновременно."""
    sem = asyncio.Semaphore(concurrency)

    async def wrapped(item):
        async with sem:
            return await generate_one(item)

    started = time.perf_counter()

    results = await asyncio.gather(
        *[wrapped(item) for item in items]
    )

    elapsed = time.perf_counter() - started

    return results, elapsed

# Расчет метрик

In [18]:
def calculate_metrics(run_results, elapsed_sec):
    """Вычисляет метрики пакетного прогона."""
    results = [result for result, _ in run_results]
    latencies = [latency for _, latency in run_results]

    n = len(results)
    valid = sum(result.valid for result in results)

    errors_by_reason = Counter(
        reason.value
        for result in results
        if not result.valid
        for reason in set(result.errors)
    )

    return {
        "n": n,
        "valid": valid,
        "valid_rate": valid / n,
        "elapsed_sec": elapsed_sec,
        "throughput": n / elapsed_sec,
        "avg_latency": sum(latencies) / n,
        "time_for_300_sec": 300 / (n / elapsed_sec),
        "avg_attempts": sum(result.attempts for result in results) / n,
        "input_tokens": sum(result.input_tokens for result in results),
        "output_tokens": sum(result.output_tokens for result in results),
        "errors_by_reason": dict(errors_by_reason),
    }

# Пакетный прогон

In [ ]:
async def test_run():
    async def generate(product):
        return await generate_timed(
            product=product,
            client=client,
            model_name=model_name,
            generation_params=generation_params,
        )

    run_results, elapsed = await run_concurrent(
        items=prepared_benchmark[:5],
        concurrency=2,
        generate_one=generate,
    )

    return calculate_metrics(run_results, elapsed)


test_metrics = await test_run()
test_metrics

In [ ]:
experiment_items = prepared_benchmark[:40]

len(experiment_items)

In [ ]:
experiment_results = []

for concurrency in [1, 4, 16, 32]:
    _, metrics = await run_benchmark(
        items=experiment_items,
        concurrency=concurrency,
        client=client,
        model_name=model_name,
        generation_params=generation_params,
    )

    experiment_results.append({
        "concurrency": concurrency,
        **metrics,
    })

    print(
        f"concurrency={concurrency}: "
        f"throughput={metrics['throughput']:.2f} card/s, "
        f"latency={metrics['avg_latency']:.2f} s, "
        f"valid={metrics['valid_rate']:.1%}"
    )

In [ ]:
experiment_df = pd.DataFrame(experiment_results)

experiment_df[
    [
        "concurrency",
        "throughput",
        "avg_latency",
        "valid_rate",
        "elapsed_sec",
        "time_for_300_sec",
        "avg_attempts",
    ]
].round(3)

In [ ]:
experiment_df.plot(
        x="concurrency",
        y="throughput",
        marker="o",
        ylabel="карточек / сек",
        title="Throughput в зависимости от concurrency",
        legend=False,
)

In [ ]:
experiment_df.plot(
        x="concurrency",
        y="avg_latency",
        marker="o",
        ylabel="секунды",
        title="Средняя latency в зависимости от concurrency",
        legend=False,
)

In [ ]:
experiment_df[
        ["concurrency", "valid_rate", "avg_attempts"]
]

# Финальный прогон

In [ ]:
selected_concurrency = 16

In [ ]:
full_results, full_metrics = await run_benchmark(
        items=prepared_benchmark,
        concurrency=selected_concurrency,
        client=client,
        model_name=model_name,
        generation_params=generation_params,
)

full_metrics

In [ ]:
print(f"Карточек: {full_metrics['n']}")
print(f"Валидных: {full_metrics['valid']}")
print(f"Доля валидных: {full_metrics['valid_rate']:.1%}")
print(f"Время: {full_metrics['elapsed_sec']:.1f} сек")
print(f"Throughput: {full_metrics['throughput']:.2f} карточек/сек")
print(f"Средняя latency: {full_metrics['avg_latency']:.2f} сек")
print(f"Среднее число попыток: {full_metrics['avg_attempts']:.2f}")

print()
print(f"Качество >= 80%: {full_metrics['valid_rate'] >= 0.80}")
print(f"Время <= 8 минут: {full_metrics['elapsed_sec'] <= 480}")

In [ ]:
full_metrics["errors_by_reason"]

In [ ]:
cost_vllm_per_1000 = (
        full_metrics["elapsed_sec"] / 3600 * 65
        / full_metrics["n"]
        * 1000
)

print(f"Стоимость 1000 карточек: {cost_vllm_per_1000:.2f} ₽")
print(f"Стоимость <= 25 ₽: {cost_vllm_per_1000 <= 25}")